<a href="https://colab.research.google.com/github/fborrasumh/optimizia/blob/main/cuadernos/03-metaheuristicas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# OptimizIA · Cuaderno 3: Heurísticas y metaheurísticas

**Técnicas Avanzadas de Optimización (8265)** · Máster en Estadística Computacional y Ciencia de Datos para la Toma de Decisiones · UMH.
Cuaderno de práctica de la app OptimizIA (Fernando Borrás Rocher, Mercedes Landete Ruiz y Juan Francisco Monge Ivars). Licencia MIT.

**Resultados de aprendizaje que trabaja:** RA5 (diseñar y evaluar heurísticas y metaheurísticas para problemas combinatorios). *Este bloque aparece en la descripción de la guía, no en sus objetivos.*

Cuando el problema es demasiado grande para un método exacto, buscamos buenas soluciones rápido. Lo importante: saber **cuánto** te alejas del óptimo (o de una cota).

> Cómo usarlo: ejecuta las celdas en orden (Mayús+Intro). Los ejercicios están marcados con ✏️. Las celdas «Solución» están plegadas: despliégalas solo después de intentarlo.
> Las instancias son **pequeñas y ficticias**, pensadas para ver el mecanismo; los tiempos que veas dependen de la máquina de Colab.

In [ ]:
# Instala OR-Tools si no está (en Colab tarda unos segundos)
try:
    import ortools
except ImportError:
    %pip install -q ortools
    import ortools
print('OR-Tools', ortools.__version__)

## 1. Viajante (TSP): construir y mejorar

Instancia de 40 ciudades al azar. Primero una heurística **constructiva** (vecino más próximo) y luego **búsqueda local 2-opt**: quitar dos aristas $(a,b),(c,e)$ y reconectar $(a,c),(b,e)$ si $\Delta = d_{ac}+d_{be}-d_{ab}-d_{ce} < 0$.

In [ ]:
import random, math, time
rnd = random.Random(11); N = 40
P = [(rnd.uniform(0, 100), rnd.uniform(0, 100)) for _ in range(N)]
D = [[round(math.dist(P[i], P[j])) for j in range(N)] for i in range(N)]
long_ = lambda t: sum(D[t[k]][t[(k + 1) % len(t)]] for k in range(len(t)))

def vecino_mas_proximo(inicio=0):
    t, libres = [inicio], set(range(N)) - {inicio}
    while libres:
        sig = min(libres, key=lambda j: D[t[-1]][j]); t.append(sig); libres.remove(sig)
    return t

def dos_opt(t):
    t, mejora = t[:], True
    while mejora:
        mejora = False
        for i in range(N - 1):
            for k in range(i + 2, N if i else N - 1):
                a, b, c, e = t[i], t[i + 1], t[k], t[(k + 1) % N]
                if D[a][c] + D[b][e] - D[a][b] - D[c][e] < 0:
                    t[i + 1:k + 1] = reversed(t[i + 1:k + 1]); mejora = True
    return t

t0 = vecino_mas_proximo(); t1 = dos_opt(t0)
print('Vecino más próximo:', long_(t0), '· tras 2-opt:', long_(t1))

## 2. GRASP: construcción aleatorizada + búsqueda local, con multiarranque

En cada paso se elige al azar entre los candidatos de la **lista restringida** (RCL): los que están a menos de $\alpha$ del mejor. $\alpha=0$ es voraz puro; $\alpha=1$, aleatorio puro.

In [ ]:
def grasp(iters=30, alfa=0.3, semilla=0):
    r = random.Random(semilla); mejor = None
    for _ in range(iters):
        t = [r.randrange(N)]; libres = set(range(N)) - set(t)
        while libres:
            dist = {j: D[t[-1]][j] for j in libres}; lo, hi = min(dist.values()), max(dist.values())
            rcl = [j for j in libres if dist[j] <= lo + alfa * (hi - lo)]
            j = r.choice(rcl); t.append(j); libres.remove(j)
        t = dos_opt(t)
        if mejor is None or long_(t) < long_(mejor): mejor = t
    return mejor
for alfa in (0.0, 0.2, 0.5, 1.0):
    print(f'GRASP α={alfa}: {long_(grasp(alfa=alfa))}')

## 3. Metaheurísticas de OR-Tools (enrutamiento)

La librería de *routing* de OR-Tools trae varias metaheurísticas: **búsqueda local guiada** (GLS, penaliza las aristas de los óptimos locales), **búsqueda tabú** y **recocido simulado**. Se les da un límite de tiempo.

In [ ]:
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

def or_tools_tsp(meta, segundos=3):
    man = pywrapcp.RoutingIndexManager(N, 1, 0); R = pywrapcp.RoutingModel(man)
    cb = R.RegisterTransitCallback(lambda a, b: D[man.IndexToNode(a)][man.IndexToNode(b)])
    R.SetArcCostEvaluatorOfAllVehicles(cb)
    p = pywrapcp.DefaultRoutingSearchParameters()
    p.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    if meta: p.local_search_metaheuristic = getattr(routing_enums_pb2.LocalSearchMetaheuristic, meta); p.time_limit.seconds = segundos
    sol = R.SolveWithParameters(p); return sol.ObjectiveValue()

res = {}
for meta in (None, 'GREEDY_DESCENT', 'GUIDED_LOCAL_SEARCH', 'TABU_SEARCH', 'SIMULATED_ANNEALING'):
    res[meta or 'solo constructiva'] = or_tools_tsp(meta)
    print(f'{meta or "solo constructiva":22s}: {res[meta or "solo constructiva"]}')

**Cuidado al comparar.** Con 3 segundos y esta instancia, algunas metaheurísticas pueden no mejorar la solución inicial y un GRASP sencillo puede ganar a todas. Eso no es una conclusión general: depende del límite de tiempo, la instancia y la semilla. Repite con `segundos=10` y otras semillas antes de opinar.

## 4. ¿A qué distancia del óptimo estamos? Una cota

Una metaheurística no certifica nada por sí sola. Para saber cuánto te alejas necesitas una **cota inferior**. Aquí usamos el **árbol 1** (*1-tree*): árbol de expansión mínima sobre las ciudades $1..N-1$ más las dos aristas más baratas de la ciudad 0. Todo ciclo hamiltoniano es un 1-árbol, así que su peso mínimo es una cota inferior (débil; Held y Karp la mejoran con multiplicadores lagrangianos: ¡el cuaderno 2 otra vez!).

In [ ]:
def cota_1_arbol(pen=None):
    pen = pen or [0] * N; Dp = lambda i, j: D[i][j] + pen[i] + pen[j]
    dentro, coste, grado = {1}, 0, [0] * N
    mejor = {j: (Dp(1, j), 1) for j in range(2, N)}
    while mejor:   # Prim sobre 1..N-1
        j = min(mejor, key=lambda k: mejor[k][0]); cj, pj = mejor.pop(j); coste += cj; grado[j] += 1; grado[pj] += 1
        for k in mejor:
            if Dp(j, k) < mejor[k][0]: mejor[k] = (Dp(j, k), j)
    dos = sorted(range(1, N), key=lambda j: Dp(0, j))[:2]
    for j in dos: coste += Dp(0, j); grado[j] += 1
    grado[0] = 2
    return coste - 2 * sum(pen), grado

LB0 = cota_1_arbol()[0]
pen, LB, paso = [0] * N, LB0, 1.0       # Held-Karp: subgradiente sobre las penalizaciones de los nodos
UB = min(min(res.values()), long_(grasp(alfa=0.5)))
for k in range(200):
    val, g = cota_1_arbol(pen); LB = max(LB, val)
    sg = [gi - 2 for gi in g]
    if not any(sg): break
    t = paso * (UB - val) / sum(s * s for s in sg); pen = [pen[i] + t * sg[i] for i in range(N)]
    if k % 40 == 39: paso /= 2
print(f'Cota 1-árbol: {LB0:.1f} · cota Held-Karp: {LB:.1f} · mejor tour: {UB} → gap ≤ {100*(UB-LB)/UB:.1f} %')

✏️ **Ejercicio.** Implementa un **recocido simulado** propio con movimientos 2-opt: acepta un empeoramiento $\Delta>0$ con probabilidad $e^{-\Delta/T}$ y enfría $T \leftarrow 0{,}995\,T$. Compara su resultado con GLS de OR-Tools y con la cota.

In [ ]:
#@title Solución del ejercicio { display-mode: "form" }
def recocido(T=50.0, frio=0.995, iters=40000, semilla=1):
    r = random.Random(semilla); t = vecino_mas_proximo(); L_ = long_(t); mejor, Lm = t[:], L_
    for _ in range(iters):
        i, k = sorted(r.sample(range(N), 2))
        if k - i < 2 or (i == 0 and k == N - 1): continue
        a, b, c, e = t[i], t[i + 1], t[k], t[(k + 1) % N]
        delta = D[a][c] + D[b][e] - D[a][b] - D[c][e]
        if delta < 0 or r.random() < math.exp(-delta / T):
            t[i + 1:k + 1] = reversed(t[i + 1:k + 1]); L_ += delta
            if L_ < Lm: mejor, Lm = t[:], L_
        T *= frio if T > 1e-3 else 1
    return Lm
print('Recocido propio:', recocido(), '· GLS OR-Tools:', res['GUIDED_LOCAL_SEARCH'], '· cota:', round(LB, 1))

## 🎯 Tu instancia personal y tu entrega

1. En OptimizIA, ve a **Cuadernos → Tu instancia personal** y pulsa **Copiar**.
2. Sustituye la línea `INSTANCIA = None` de la celda siguiente por la que has copiado y ejecuta las dos celdas.
3. Copia la línea `ENTREGA = {…}` que aparece y pégala en OptimizIA: la app recalcula tu instancia con su propio solver y valida tu resultado.

> ✏️ Antes de ejecutar, **predice**: ¿a qué distancia de la cota de Held-Karp crees que quedará tu mejor ruta?

In [ ]:
INSTANCIA = None   # ← sustituye esta línea por la que copias de OptimizIA

In [ ]:
import json, math
def mejor_ruta(P, segundos=10):
    n = len(P); Dm = [[round(math.dist(P[i], P[j])) for j in range(n)] for i in range(n)]
    man = pywrapcp.RoutingIndexManager(n, 1, 0); R = pywrapcp.RoutingModel(man)
    cb = R.RegisterTransitCallback(lambda a, b: Dm[man.IndexToNode(a)][man.IndexToNode(b)]); R.SetArcCostEvaluatorOfAllVehicles(cb)
    p = pywrapcp.DefaultRoutingSearchParameters(); p.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    p.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH; p.time_limit.seconds = segundos
    sol = R.SolveWithParameters(p); i = R.Start(0); ruta = []
    while not R.IsEnd(i): ruta.append(man.IndexToNode(i)); i = sol.Value(R.NextVar(i))
    return ruta, sum(Dm[ruta[k]][ruta[(k + 1) % n]] for k in range(n))
if INSTANCIA:
    ruta, L = mejor_ruta(INSTANCIA['ciudades'])
    print('Longitud:', L, '· ✏️ mejórala con tu recocido o tu GRASP si puedes')
    print('ENTREGA =', json.dumps({'app': 'optimizia', 'cuaderno': 3, 'semilla': INSTANCIA['semilla'], 'res': {'ruta': ruta, 'longitud': L}}))
else:
    print('Pega primero tu INSTANCIA en la celda anterior.')

## 🏆 Torneo de la semana

Toda la clase tiene la misma instancia de 30 ciudades, distinta cada semana. En OptimizIA, ve a **Torneo** y copia la línea `TORNEO = {…}`. Ejecuta y pega la ruta en la app: verá su longitud y su gap frente a la cota de Held-Karp. Vale todo: GLS, tu recocido, GRASP… y más segundos.

In [ ]:
TORNEO = None   # ← sustituye esta línea por la que copias de OptimizIA (Torneo)
if TORNEO:
    ruta, L = mejor_ruta(TORNEO['ciudades'], segundos=15)
    print('Longitud:', L)
    print('RUTA =', json.dumps({'app': 'optimizia', 'torneo': TORNEO['semana'], 'ruta': ruta}))

## Para llevarte
- Constructiva → búsqueda local → metaheurística (escapar de óptimos locales: tabú, recocido, GLS, GRASP multiarranque).
- Compara siempre con la misma instancia, el mismo tiempo y varias semillas.
- Sin una cota (relajación, 1-árbol, Held-Karp) no sabes cuánto te queda por mejorar.
- Bibliografía complementaria de la guía: Martí y Reinelt, *The Linear Ordering Problem* (heurísticas y metaheurísticas en un problema concreto).